In [ ]:
import matplotlib.cm as cm
from matplotlib.colors import hsv_to_rgb, Normalize
import matplotlib.pyplot as plt
import imageio
import numpy as np

In [ ]:
def disparity_map_to_image(disparity, mapper=None):
    """
    Convert a disparity map to an RGB image.
    Source: https://github.com/uzh-rpg/DSEC/blob/main/scripts/dataset/visualization.py

    Args:
        disparity (np.ndarray): Disparity map with shape (1, height, width).

    Returns:
        np.ndarray: RGB image of the disparity map.
    """

    # check shape
    assert disparity.ndim == 3 and disparity.shape[
        0] == 1, "Disparity must have shape (1, height, width)."

    # disparity magnitude for nonzero pixels
    disparity = disparity.squeeze(0)  # remove channel
    disp_pixels = np.argwhere(disparity > 0)
    y, x = disp_pixels[:, 0], disp_pixels[:, 1]
    disp_valid = disparity[y, x]
    min_disp = disp_valid.min() if len(disp_valid) > 0 else 0
    max_disp = disp_valid.max() if len(disp_valid) > 0 else 0

    # disparity colormap (in reverse if depth map)
    norm = Normalize(vmin=min_disp, vmax=max_disp, clip=True)
    if mapper is None:
        mapper = cm.ScalarMappable(norm=norm, cmap="inferno")
    disp_color = mapper.to_rgba(disp_valid)[..., :3]
    image = np.zeros((*disparity.shape, 3))

    # to rgb ints
    image[y, x] = disp_color
    image = (image * 255).astype(np.uint8)

    return image, mapper


In [ ]:
from collections import OrderedDict
sequences = OrderedDict({
    "interlaken_00_a(540)": ("il0a", 540),
    # "interlaken_00_b(500)": ("il0b", 500),
    "interlaken_00_b(560)": ("il0b", 560),
    "interlaken_01_a(1680)": ("il1a", 1680),
    "thun_01_b(400)": ("t1b", 400),
    "zurich_city_12_a(440)": ("zc12a", 440),
    "zurich_city_13_a(260)": ("zc13a", 260),
})

In [ ]:
fig, axs = plt.subplots(len(sequences), 3, figsize=(15, 25), constrained_layout=True)
titles = ["Image", "Cho et al. ", "Ours"]
for i, (name, (sequence, frame)) in enumerate(sequences.items()):
    for j in range(3):
        ax = axs[i, j]
        if j == 0:
            ax.set_ylabel(name, fontsize=20, labelpad=15)
        if i == 0:
            ax.set_title(titles[j], fontsize=30, pad=15)
        ax.tick_params(
            axis='both',          # changes apply to the x-axis
            which='both',      # both major and minor ticks are affected
            left=False,      # ticks along the bottom edge are off
            top=False,
            bottom=False,  # ticks along the top edge are off
            labelbottom=False,
            labelleft=False)
        
        if j == 0:
            # image
            ax.imshow(plt.imread(f"dsec_qual/{sequence}_{str(frame).zfill(6)}i.png"))


        if j == 1:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame-1).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, mapper = disparity_map_to_image(np.expand_dims(disparity, 0))
            ax.imshow(disparity_img)

        if j == 2:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, _ = disparity_map_to_image(np.expand_dims(disparity, 0), mapper)
            ax.imshow(disparity_img)

fig.savefig('dsec.pdf')

In [ ]:
fig, axs = plt.subplots(3, len(sequences), figsize=(
    25, 10), constrained_layout=True)
titles = ["Image", "Cho et al. ", "Ours"]
for i, (name, (sequence, frame)) in enumerate(sequences.items()):
    for j in range(3):
        ax = axs[j, i]
        if j == 0:
            ax.set_title(name, fontsize=20, pad=15)
        if i == 0:
            ax.set_ylabel(titles[j], fontsize=30, labelpad=15)
        ax.tick_params(
            axis='both',          # changes apply to the x-axis
            which='both',      # both major and minor ticks are affected
            left=False,      # ticks along the bottom edge are off
            top=False,
            bottom=False,  # ticks along the top edge are off
            labelbottom=False,
            labelleft=False)

        if j == 0:
            # image
            ax.imshow(plt.imread(
                f"dsec_qual/{sequence}_{str(frame).zfill(6)}i.png"))

        if j == 1:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame-1).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, mapper = disparity_map_to_image(
                np.expand_dims(disparity, 0))
            ax.imshow(disparity_img)

        if j == 2:
            # flow
            disparity = imageio.imread(
                f"dsec_qual/{sequence}_{str(frame).zfill(6)}.png").astype(np.float32) / 256
            disparity_img, _ = disparity_map_to_image(
                np.expand_dims(disparity, 0), mapper)
            ax.imshow(disparity_img)

fig.savefig('dsec-landscape.pdf')